# Analisis exploratorio con SQL: Indicadores socioeconomicos de Colombia

En este notebook vamos a explorar datos socioeconomicos de Colombia -- poblacion, pobreza, educacion y empleo -- usando SQL puro desde Python con DuckDB.

La razon por la que uso SQL para esto es simple: SQL sigue siendo el lenguaje mas importante para trabajar con datos. Toda empresa lo usa, desde startups hasta bancos. Y con herramientas como DuckDB, podemos ejecutar consultas analiticas complejas sin necesitar un servidor de base de datos -- todo corre en memoria dentro de nuestro notebook.

Los datos que vamos a usar estan inspirados en datasets reales del DANE y otras entidades del gobierno colombiano: poblacion por departamento, indicadores de pobreza, metricas de educacion y datos del mercado laboral.

Lo que vamos a cubrir va desde consultas basicas hasta window functions avanzadas, CTEs, subqueries correlacionadas y analisis multi-tabla. Al final, construimos un scorecard integral a nivel departamental que combina las 4 fuentes de datos.

Vamos a seguir esta ruta:

1. Preparacion de datos -- crear las 4 tablas en DuckDB
2. Consultas basicas -- SELECT, WHERE, GROUP BY, HAVING
3. JOINs -- conectar tablas para responder preguntas mas complejas
4. Window Functions -- calculos avanzados sin perder el detalle fila por fila
5. CTEs y Subqueries -- dividir problemas complejos en pasos
6. Consultas avanzadas -- combinar todo para analisis profundo
7. Visualizacion -- los numeros cuentan mejor con graficos

In [ ]:
!pip install duckdb pandas matplotlib seaborn -q

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 11})

con = duckdb.connect(":memory:")
print(f"DuckDB {duckdb.__version__} connected (in-memory)")

## Preparacion de datos

Lo primero que hacemos es crear las 4 tablas que vamos a analizar. Estan modeladas a partir de datasets del DANE (la entidad de estadisticas de Colombia): poblacion, pobreza, matricula educativa y mercado laboral.

Son 33 departamentos y 6 anos (2019-2024). La razon por la que uso este rango es que incluye el choque del COVID en 2020, lo cual nos da patrones interesantes para analizar con SQL.

Vamos a registrar los DataFrames como tablas en DuckDB para poder hacer SQL directamente sobre ellos.

In [ ]:
np.random.seed(42)

# Department definitions with region, capital, and approximate 2019 population
dept_info = {
    "Amazonas":            ("Amazonia",   "Leticia",            80000),
    "Antioquia":           ("Andina",     "Medellin",         6800000),
    "Arauca":              ("Orinoquia",  "Arauca",            290000),
    "Atlantico":           ("Caribe",     "Barranquilla",     2600000),
    "Bogota D.C.":         ("Andina",     "Bogota",           7700000),
    "Bolivar":             ("Caribe",     "Cartagena",        2200000),
    "Boyaca":              ("Andina",     "Tunja",            1280000),
    "Caldas":              ("Andina",     "Manizales",         998000),
    "Caqueta":             ("Amazonia",   "Florencia",         510000),
    "Casanare":            ("Orinoquia",  "Yopal",             430000),
    "Cauca":               ("Pacifica",   "Popayan",          1470000),
    "Cesar":               ("Caribe",     "Valledupar",       1290000),
    "Choco":               ("Pacifica",   "Quibdo",            540000),
    "Cordoba":             ("Caribe",     "Monteria",         1850000),
    "Cundinamarca":        ("Andina",     "Bogota",           2950000),
    "Guainia":             ("Orinoquia",  "Inirida",            50000),
    "Guaviare":            ("Orinoquia",  "San Jose del Guaviare", 115000),
    "Huila":               ("Andina",     "Neiva",            1200000),
    "La Guajira":          ("Caribe",     "Riohacha",         1100000),
    "Magdalena":           ("Caribe",     "Santa Marta",      1380000),
    "Meta":                ("Orinoquia",  "Villavicencio",    1060000),
    "Narino":              ("Pacifica",   "Pasto",            1800000),
    "Norte de Santander":  ("Andina",     "Cucuta",           1490000),
    "Putumayo":            ("Amazonia",   "Mocoa",             380000),
    "Quindio":             ("Andina",     "Armenia",           580000),
    "Risaralda":           ("Andina",     "Pereira",           970000),
    "San Andres":          ("Caribe",     "San Andres",         82000),
    "Santander":           ("Andina",     "Bucaramanga",      2190000),
    "Sucre":               ("Caribe",     "Sincelejo",         910000),
    "Tolima":              ("Andina",     "Ibague",           1420000),
    "Valle del Cauca":     ("Pacifica",   "Cali",             4760000),
    "Vaupes":              ("Amazonia",   "Mitu",               46000),
    "Vichada":             ("Orinoquia",  "Puerto Carreno",    115000),
}

years = list(range(2019, 2025))
departments = list(dept_info.keys())

# Base poverty rates per department (approximate DANE figures)
base_poverty = {
    "Choco": 0.63, "La Guajira": 0.60, "Cauca": 0.54, "Narino": 0.50,
    "Cordoba": 0.52, "Sucre": 0.50, "Magdalena": 0.49, "Bolivar": 0.46,
    "Cesar": 0.47, "Vaupes": 0.55, "Guainia": 0.52, "Vichada": 0.50,
    "Guaviare": 0.45, "Amazonas": 0.44, "Caqueta": 0.43, "Putumayo": 0.42,
    "Arauca": 0.38, "Huila": 0.40, "Norte de Santander": 0.43, "Tolima": 0.37,
    "San Andres": 0.35, "Meta": 0.32, "Boyaca": 0.36, "Caldas": 0.33,
    "Casanare": 0.30, "Atlantico": 0.35, "Quindio": 0.32, "Risaralda": 0.30,
    "Santander": 0.28, "Antioquia": 0.30, "Valle del Cauca": 0.29,
    "Cundinamarca": 0.26, "Bogota D.C.": 0.24,
}

print(f"Departments: {len(departments)}, Years: {years}")

In [ ]:
# --- Table 1: poblacion (33 depts x 6 years = 198 rows) ---
rows_pob = []
for dept in departments:
    region, capital, pop2019 = dept_info[dept]
    for i, yr in enumerate(years):
        growth = 1 + np.random.uniform(0.008, 0.015) * i
        pop = int(pop2019 * growth + np.random.normal(0, pop2019 * 0.005))
        rows_pob.append((dept, yr, pop, capital, region))

df_poblacion = pd.DataFrame(rows_pob, columns=["departamento", "ano", "poblacion", "capital", "region"])

# --- Table 2: pobreza (33 depts x 6 years = 198 rows) ---
rows_pov = []
for dept in departments:
    bp = base_poverty[dept]
    for yr in years:
        covid_shock = 0.07 if yr == 2020 else (0.03 if yr == 2021 else 0)
        trend = -0.008 * (yr - 2019)  # gradual improvement
        inc = np.clip(bp + trend + covid_shock + np.random.normal(0, 0.015), 0.10, 0.75)
        ext = np.clip(inc * np.random.uniform(0.25, 0.40), 0.03, 0.35)
        gini = np.clip(0.35 + inc * 0.35 + np.random.normal(0, 0.01), 0.40, 0.65)
        rows_pov.append((dept, yr, round(inc, 4), round(ext, 4), round(gini, 4)))

df_pobreza = pd.DataFrame(rows_pov, columns=["departamento", "ano", "incidencia_pobreza", "pobreza_extrema", "gini"])

# --- Table 3: educacion (33 depts x 6 years x 4 levels = 792 rows) ---
niveles = ["Preescolar", "Basica Primaria", "Basica Secundaria", "Media"]
rows_edu = []
for dept in departments:
    _, _, pop = dept_info[dept]
    for yr in years:
        for niv in niveles:
            base_mat = {"Preescolar": 0.04, "Basica Primaria": 0.12,
                        "Basica Secundaria": 0.08, "Media": 0.05}[niv]
            mat = int(pop * base_mat * (1 + np.random.uniform(-0.05, 0.05)))
            tasa_des = np.clip(np.random.uniform(0.02, 0.08) + (0.03 if yr == 2020 else 0), 0.01, 0.15)
            des = int(mat * tasa_des)
            rows_edu.append((dept, yr, niv, mat, des, round(tasa_des, 4)))

df_educacion = pd.DataFrame(rows_edu, columns=["departamento", "ano", "nivel_educativo", "matriculados", "desertores", "tasa_desercion"])

# --- Table 4: empleo (33 depts x 5 years x 4 quarters = 660 rows) ---
rows_emp = []
for dept in departments:
    bp = base_poverty[dept]
    base_unemp = 0.08 + bp * 0.15  # higher poverty -> higher unemployment
    for yr in years[:5]:  # 2019-2023 (quarterly)
        for q in range(1, 5):
            covid_q = 0.08 if (yr == 2020 and q == 2) else (0.04 if yr == 2020 else 0)
            unemp = np.clip(base_unemp + covid_q + np.random.normal(0, 0.012), 0.05, 0.25)
            occup = np.clip(1 - unemp - np.random.uniform(0.30, 0.42), 0.35, 0.65)
            inform = np.clip(0.40 + bp * 0.45 + np.random.normal(0, 0.03), 0.25, 0.85)
            rows_emp.append((dept, yr, q, round(unemp, 4), round(occup, 4), round(inform, 4)))

df_empleo = pd.DataFrame(rows_emp, columns=["departamento", "ano", "trimestre", "tasa_desempleo", "tasa_ocupacion", "tasa_informalidad"])

print(f"poblacion:  {df_poblacion.shape}")
print(f"pobreza:    {df_pobreza.shape}")
print(f"educacion:  {df_educacion.shape}")
print(f"empleo:     {df_empleo.shape}")

In [ ]:
# Register DataFrames as DuckDB tables
con.register("poblacion", df_poblacion)
con.register("pobreza", df_pobreza)
con.register("educacion", df_educacion)
con.register("empleo", df_empleo)

# Verify tables
con.sql("SHOW TABLES").df()

## Consultas basicas -- Empezar por lo fundamental: SELECT, WHERE, GROUP BY

Vamos a empezar con lo mas basico de SQL. Lo que hacemos aqui es familiarizarnos con la estructura de los datos y sacar las primeras estadisticas descriptivas.

Lo que hay que esperar: consultas simples que filtran, agrupan y resumen. Si ya sabes SQL, esto es calentamiento. Si estas empezando, presta atencion a la estructura porque todo lo que viene despues se construye sobre esto.

In [ ]:
# Top 10 departments by population in 2024
con.sql("""
    SELECT departamento, region, poblacion
    FROM poblacion
    WHERE ano = 2024
    ORDER BY poblacion DESC
    LIMIT 10
""").df()

In [ ]:
# Population by region with multiple aggregations
con.sql("""
    SELECT
        region,
        COUNT(DISTINCT departamento) AS n_departamentos,
        SUM(poblacion)               AS poblacion_total,
        AVG(poblacion)::INT          AS poblacion_promedio,
        MIN(poblacion)               AS poblacion_min,
        MAX(poblacion)               AS poblacion_max
    FROM poblacion
    WHERE ano = 2024
    GROUP BY region
    ORDER BY poblacion_total DESC
""").df()

In [ ]:
# HAVING: Regions where average poverty incidence exceeds 40%
con.sql("""
    SELECT
        p.region,
        ROUND(AVG(pov.incidencia_pobreza) * 100, 1) AS avg_pobreza_pct,
        ROUND(AVG(pov.gini), 3)                      AS avg_gini
    FROM pobreza pov
    JOIN poblacion p USING (departamento, ano)
    WHERE pov.ano = 2024
    GROUP BY p.region
    HAVING AVG(pov.incidencia_pobreza) > 0.40
    ORDER BY avg_pobreza_pct DESC
""").df()

In [ ]:
# DISTINCT: Count unique department-level combinations in education data
con.sql("""
    SELECT
        COUNT(DISTINCT departamento)                           AS departamentos,
        COUNT(DISTINCT nivel_educativo)                        AS niveles,
        COUNT(DISTINCT (departamento, ano, nivel_educativo))   AS combinaciones_unicas,
        COUNT(*)                                               AS total_filas
    FROM educacion
""").df()

## JOINs -- Conectar tablas para responder preguntas mas complejas

Hasta ahora hemos trabajado con una tabla a la vez. Pero las preguntas interesantes requieren cruzar informacion de multiples fuentes.

Lo que vamos a ver aqui son 4 tipos de JOINs:
- **INNER JOIN**: solo trae filas que existen en ambas tablas
- **LEFT JOIN**: trae todo de la tabla izquierda aunque no haya match a la derecha (aparecen NULLs)
- **Multi-table JOIN**: conectar 3 o mas tablas en una sola consulta
- **Self-join**: unir una tabla consigo misma (util para comparar ano contra ano)

La razon por la que esto importa tanto: en la vida real los datos nunca estan en una sola tabla. Saber hacer JOINs bien es probablemente la habilidad mas valiosa de SQL.

In [ ]:
# INNER JOIN: Population-weighted poverty by region (2024)
con.sql("""
    SELECT
        p.region,
        SUM(p.poblacion)                                                  AS poblacion_total,
        ROUND(SUM(pov.incidencia_pobreza * p.poblacion)
              / SUM(p.poblacion) * 100, 2)                                AS pobreza_ponderada_pct,
        ROUND(SUM(pov.gini * p.poblacion) / SUM(p.poblacion), 3)         AS gini_ponderado
    FROM poblacion p
    INNER JOIN pobreza pov
        ON p.departamento = pov.departamento AND p.ano = pov.ano
    WHERE p.ano = 2024
    GROUP BY p.region
    ORDER BY pobreza_ponderada_pct DESC
""").df()

In [ ]:
# LEFT JOIN with NULL handling: departments and their total enrollment
# empleo only has 2019-2023, so 2024 rows will show NULL for employment
con.sql("""
    SELECT
        p.departamento,
        p.ano,
        p.poblacion,
        e.tasa_desempleo,
        COALESCE(e.tasa_desempleo, -1) AS desempleo_filled,
        CASE WHEN e.tasa_desempleo IS NULL THEN 'No data' ELSE 'Available' END AS status
    FROM poblacion p
    LEFT JOIN (
        SELECT departamento, ano, AVG(tasa_desempleo) AS tasa_desempleo
        FROM empleo GROUP BY departamento, ano
    ) e ON p.departamento = e.departamento AND p.ano = e.ano
    WHERE p.departamento = 'Bogota D.C.'
    ORDER BY p.ano
""").df()

In [ ]:
# Multi-table JOIN: comprehensive socioeconomic view (2023)
con.sql("""
    SELECT
        p.departamento,
        p.region,
        p.poblacion,
        ROUND(pov.incidencia_pobreza * 100, 1) AS pobreza_pct,
        ROUND(pov.gini, 3)                     AS gini,
        ROUND(emp.avg_desempleo * 100, 1)       AS desempleo_pct,
        ROUND(emp.avg_informalidad * 100, 1)     AS informalidad_pct,
        edu.total_matriculados
    FROM poblacion p
    JOIN pobreza pov USING (departamento, ano)
    JOIN (
        SELECT departamento, ano,
               AVG(tasa_desempleo) AS avg_desempleo,
               AVG(tasa_informalidad) AS avg_informalidad
        FROM empleo GROUP BY departamento, ano
    ) emp USING (departamento, ano)
    JOIN (
        SELECT departamento, ano, SUM(matriculados) AS total_matriculados
        FROM educacion GROUP BY departamento, ano
    ) edu USING (departamento, ano)
    WHERE p.ano = 2023
    ORDER BY p.poblacion DESC
    LIMIT 10
""").df()

In [ ]:
# Self-join: year-over-year population change
con.sql("""
    SELECT
        curr.departamento,
        curr.ano,
        curr.poblacion                                          AS poblacion_actual,
        prev.poblacion                                          AS poblacion_anterior,
        curr.poblacion - prev.poblacion                         AS cambio_absoluto,
        ROUND((curr.poblacion - prev.poblacion) * 100.0
              / prev.poblacion, 2)                              AS cambio_pct
    FROM poblacion curr
    JOIN poblacion prev
        ON curr.departamento = prev.departamento
        AND curr.ano = prev.ano + 1
    WHERE curr.ano = 2024
    ORDER BY cambio_pct DESC
    LIMIT 10
""").df()

## Window Functions -- Calculos avanzados sin perder el detalle fila por fila

Aqui es donde SQL se pone realmente interesante. Las window functions nos permiten hacer calculos sobre un grupo de filas (una "ventana") sin colapsar los datos como hace GROUP BY.

Lo que vamos a ver:
- **ROW_NUMBER, RANK, DENSE_RANK**: para crear rankings dentro de grupos
- **LAG y LEAD**: para acceder a la fila anterior o siguiente (perfecto para cambios ano contra ano)
- **SUM y AVG acumulados**: totales corridos y promedios moviles
- **NTILE**: para dividir los datos en cuartiles o percentiles

La razon por la que uso window functions en vez de subqueries para estas tareas: son mas legibles, mas eficientes y se pueden combinar varias en la misma consulta.

In [ ]:
# ROW_NUMBER: Rank departments by poverty within each region (2024)
con.sql("""
    SELECT
        p.region,
        pov.departamento,
        ROUND(pov.incidencia_pobreza * 100, 1) AS pobreza_pct,
        ROW_NUMBER() OVER (
            PARTITION BY p.region
            ORDER BY pov.incidencia_pobreza DESC
        ) AS rank_in_region
    FROM pobreza pov
    JOIN poblacion p USING (departamento, ano)
    WHERE pov.ano = 2024
    QUALIFY rank_in_region <= 3
    ORDER BY p.region, rank_in_region
""").df()

In [ ]:
# RANK vs DENSE_RANK: compare ranking methods on unemployment (2023-Q4)
con.sql("""
    SELECT
        departamento,
        ROUND(tasa_desempleo * 100, 1) AS desempleo_pct,
        RANK()       OVER (ORDER BY tasa_desempleo DESC) AS rank_val,
        DENSE_RANK() OVER (ORDER BY tasa_desempleo DESC) AS dense_rank_val,
        ROW_NUMBER() OVER (ORDER BY tasa_desempleo DESC) AS row_num
    FROM empleo
    WHERE ano = 2023 AND trimestre = 4
    ORDER BY rank_val
    LIMIT 15
""").df()

In [ ]:
# LAG/LEAD: Year-over-year change in poverty
con.sql("""
    SELECT
        departamento,
        ano,
        ROUND(incidencia_pobreza * 100, 1) AS pobreza_pct,
        ROUND(LAG(incidencia_pobreza) OVER (
            PARTITION BY departamento ORDER BY ano
        ) * 100, 1) AS pobreza_anterior,
        ROUND((incidencia_pobreza - LAG(incidencia_pobreza) OVER (
            PARTITION BY departamento ORDER BY ano
        )) * 100, 1) AS cambio_pp,
        ROUND(LEAD(incidencia_pobreza) OVER (
            PARTITION BY departamento ORDER BY ano
        ) * 100, 1) AS pobreza_siguiente
    FROM pobreza
    WHERE departamento IN ('Bogota D.C.', 'Choco', 'Antioquia')
    ORDER BY departamento, ano
""").df()

In [ ]:
# Running SUM and Moving AVG: cumulative population and 3-year moving average poverty
con.sql("""
    SELECT
        ano,
        SUM(poblacion) AS poblacion_nacional,
        SUM(SUM(poblacion)) OVER (ORDER BY ano) AS poblacion_acumulada,
        ROUND(AVG(AVG(pov.incidencia_pobreza)) OVER (
            ORDER BY ano ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) * 100, 2) AS media_movil_pobreza_3a
    FROM poblacion p
    JOIN pobreza pov USING (departamento, ano)
    GROUP BY ano
    ORDER BY ano
""").df()

In [ ]:
# NTILE: Divide departments into poverty quartiles (2024)
con.sql("""
    SELECT
        departamento,
        ROUND(incidencia_pobreza * 100, 1) AS pobreza_pct,
        NTILE(4) OVER (ORDER BY incidencia_pobreza) AS cuartil,
        CASE NTILE(4) OVER (ORDER BY incidencia_pobreza)
            WHEN 1 THEN 'Q1 - Lowest poverty'
            WHEN 2 THEN 'Q2 - Below median'
            WHEN 3 THEN 'Q3 - Above median'
            WHEN 4 THEN 'Q4 - Highest poverty'
        END AS label
    FROM pobreza
    WHERE ano = 2024
    ORDER BY incidencia_pobreza
""").df()

## CTEs y Subqueries -- Dividir problemas complejos en pasos

Cuando una consulta se vuelve muy larga o complicada, lo que hacemos es dividirla en pasos usando CTEs (Common Table Expressions). Un CTE es basicamente una tabla temporal que solo existe dentro de la consulta.

Lo que vamos a ver:
- **CTE simple**: pre-agregar datos antes de hacer un JOIN
- **CTEs anidados**: multiples pasos secuenciales, cada uno construyendo sobre el anterior
- **Subquery correlacionada**: una subconsulta que se ejecuta una vez por cada fila de la consulta principal
- **CTE + window function**: combinar ambas herramientas para analisis complejos

La razon por la que prefiero CTEs sobre subqueries anidados: son mucho mas faciles de leer y de depurar. Puedes ejecutar cada CTE por separado para verificar que devuelve lo esperado.

In [ ]:
# Simple CTE: pre-aggregate education, then join with poverty
con.sql("""
    WITH edu_agg AS (
        SELECT
            departamento, ano,
            SUM(matriculados)  AS total_matriculados,
            SUM(desertores)    AS total_desertores,
            ROUND(SUM(desertores) * 100.0 / SUM(matriculados), 2) AS tasa_desercion_global
        FROM educacion
        GROUP BY departamento, ano
    )
    SELECT
        e.departamento,
        e.ano,
        e.total_matriculados,
        e.tasa_desercion_global,
        ROUND(pov.incidencia_pobreza * 100, 1) AS pobreza_pct
    FROM edu_agg e
    JOIN pobreza pov USING (departamento, ano)
    WHERE e.ano = 2024
    ORDER BY e.tasa_desercion_global DESC
    LIMIT 10
""").df()

In [ ]:
# Nested CTEs: multi-step analysis — regional poverty trend with national comparison
con.sql("""
    WITH regional_avg AS (
        SELECT
            p.region,
            pov.ano,
            ROUND(AVG(pov.incidencia_pobreza) * 100, 2) AS avg_pobreza_region
        FROM pobreza pov
        JOIN poblacion p USING (departamento, ano)
        GROUP BY p.region, pov.ano
    ),
    national_avg AS (
        SELECT
            ano,
            ROUND(AVG(incidencia_pobreza) * 100, 2) AS avg_pobreza_nacional
        FROM pobreza
        GROUP BY ano
    )
    SELECT
        r.region,
        r.ano,
        r.avg_pobreza_region,
        n.avg_pobreza_nacional,
        ROUND(r.avg_pobreza_region - n.avg_pobreza_nacional, 2) AS diferencia_pp
    FROM regional_avg r
    JOIN national_avg n USING (ano)
    ORDER BY r.region, r.ano
""").df()

In [ ]:
# Correlated subquery: departments above their regional average poverty (2024)
con.sql("""
    SELECT
        pov.departamento,
        p.region,
        ROUND(pov.incidencia_pobreza * 100, 1) AS pobreza_pct,
        (
            SELECT ROUND(AVG(pov2.incidencia_pobreza) * 100, 1)
            FROM pobreza pov2
            JOIN poblacion p2 USING (departamento, ano)
            WHERE p2.region = p.region AND pov2.ano = pov.ano
        ) AS avg_region_pct
    FROM pobreza pov
    JOIN poblacion p USING (departamento, ano)
    WHERE pov.ano = 2024
      AND pov.incidencia_pobreza > (
          SELECT AVG(pov3.incidencia_pobreza)
          FROM pobreza pov3
          JOIN poblacion p3 USING (departamento, ano)
          WHERE p3.region = p.region AND pov3.ano = pov.ano
      )
    ORDER BY pobreza_pct DESC
""").df()

In [ ]:
# CTE + window function combo: rank departments by improvement in poverty (2019 vs 2024)
con.sql("""
    WITH delta AS (
        SELECT
            p19.departamento,
            pop.region,
            ROUND(p19.incidencia_pobreza * 100, 1) AS pobreza_2019,
            ROUND(p24.incidencia_pobreza * 100, 1) AS pobreza_2024,
            ROUND((p19.incidencia_pobreza - p24.incidencia_pobreza) * 100, 1) AS mejora_pp
        FROM pobreza p19
        JOIN pobreza p24 ON p19.departamento = p24.departamento
            AND p19.ano = 2019 AND p24.ano = 2024
        JOIN poblacion pop ON pop.departamento = p19.departamento AND pop.ano = 2024
    )
    SELECT
        departamento,
        region,
        pobreza_2019,
        pobreza_2024,
        mejora_pp,
        RANK() OVER (ORDER BY mejora_pp DESC) AS rank_nacional,
        RANK() OVER (PARTITION BY region ORDER BY mejora_pp DESC) AS rank_regional
    FROM delta
    ORDER BY mejora_pp DESC
    LIMIT 15
""").df()

## Consultas avanzadas -- Combinar todo para analisis profundo

Aqui es donde juntamos todo lo que hemos visto: JOINs, window functions, CTEs, CASE WHEN, agregaciones condicionales y UNION ALL.

Lo que hay que esperar: consultas mas largas y complejas que responden preguntas de negocio reales. Por ejemplo, que departamentos mejoraron en todos los indicadores simultaneamente, o como construir un indice compuesto de bienestar que combine pobreza, empleo y educacion.

La razon por la que incluyo estas consultas es que son el tipo de analisis que te piden en entrevistas de datos y en proyectos reales. No es suficiente saber la sintaxis -- hay que saber combinar las herramientas para responder preguntas complejas.

In [ ]:
# CASE WHEN: categorize departments by poverty level (2024)
con.sql("""
    SELECT
        departamento,
        ROUND(incidencia_pobreza * 100, 1) AS pobreza_pct,
        CASE
            WHEN incidencia_pobreza >= 0.50 THEN 'Critical'
            WHEN incidencia_pobreza >= 0.35 THEN 'High'
            WHEN incidencia_pobreza >= 0.25 THEN 'Moderate'
            ELSE 'Low'
        END AS categoria,
        CASE
            WHEN incidencia_pobreza >= 0.50 THEN 4
            WHEN incidencia_pobreza >= 0.35 THEN 3
            WHEN incidencia_pobreza >= 0.25 THEN 2
            ELSE 1
        END AS prioridad
    FROM pobreza
    WHERE ano = 2024
    ORDER BY prioridad DESC, incidencia_pobreza DESC
""").df()

In [ ]:
# PIVOT-style: poverty by region and year using conditional aggregation
con.sql("""
    SELECT
        p.region,
        ROUND(AVG(CASE WHEN pov.ano = 2019 THEN pov.incidencia_pobreza END) * 100, 1) AS "2019",
        ROUND(AVG(CASE WHEN pov.ano = 2020 THEN pov.incidencia_pobreza END) * 100, 1) AS "2020",
        ROUND(AVG(CASE WHEN pov.ano = 2021 THEN pov.incidencia_pobreza END) * 100, 1) AS "2021",
        ROUND(AVG(CASE WHEN pov.ano = 2022 THEN pov.incidencia_pobreza END) * 100, 1) AS "2022",
        ROUND(AVG(CASE WHEN pov.ano = 2023 THEN pov.incidencia_pobreza END) * 100, 1) AS "2023",
        ROUND(AVG(CASE WHEN pov.ano = 2024 THEN pov.incidencia_pobreza END) * 100, 1) AS "2024",
        ROUND((AVG(CASE WHEN pov.ano = 2019 THEN pov.incidencia_pobreza END)
             - AVG(CASE WHEN pov.ano = 2024 THEN pov.incidencia_pobreza END)) * 100, 1) AS delta_pp
    FROM pobreza pov
    JOIN poblacion p USING (departamento, ano)
    GROUP BY p.region
    ORDER BY delta_pp DESC
""").df()

In [ ]:
# UNION ALL: combine key indicators into a single long-format view
con.sql("""
    SELECT departamento, ano, 'Pobreza (%)'      AS indicador,
           ROUND(incidencia_pobreza * 100, 1)     AS valor
    FROM pobreza WHERE ano = 2024

    UNION ALL

    SELECT departamento, ano, 'Desempleo (%)'     AS indicador,
           ROUND(AVG(tasa_desempleo) * 100, 1)    AS valor
    FROM empleo WHERE ano = 2023
    GROUP BY departamento, ano

    UNION ALL

    SELECT departamento, ano, 'Informalidad (%)' AS indicador,
           ROUND(AVG(tasa_informalidad) * 100, 1) AS valor
    FROM empleo WHERE ano = 2023
    GROUP BY departamento, ano

    ORDER BY departamento, indicador
    LIMIT 20
""").df()

In [ ]:
# Which departments improved most across ALL indicators simultaneously?
con.sql("""
    WITH poverty_delta AS (
        SELECT departamento,
               MAX(CASE WHEN ano = 2019 THEN incidencia_pobreza END)
             - MAX(CASE WHEN ano = 2024 THEN incidencia_pobreza END) AS mejora_pobreza
        FROM pobreza GROUP BY departamento
    ),
    unemployment_delta AS (
        SELECT departamento,
               AVG(CASE WHEN ano = 2019 THEN tasa_desempleo END)
             - AVG(CASE WHEN ano = 2023 THEN tasa_desempleo END) AS mejora_desempleo
        FROM empleo GROUP BY departamento
    ),
    desertion_delta AS (
        SELECT departamento,
               AVG(CASE WHEN ano = 2019 THEN tasa_desercion END)
             - AVG(CASE WHEN ano = 2024 THEN tasa_desercion END) AS mejora_desercion
        FROM educacion GROUP BY departamento
    )
    SELECT
        pd.departamento,
        ROUND(pd.mejora_pobreza * 100, 2)     AS mejora_pobreza_pp,
        ROUND(ud.mejora_desempleo * 100, 2)    AS mejora_desempleo_pp,
        ROUND(dd.mejora_desercion * 100, 2)    AS mejora_desercion_pp,
        ROUND((pd.mejora_pobreza + ud.mejora_desempleo + dd.mejora_desercion) * 100, 2) AS score_compuesto
    FROM poverty_delta pd
    JOIN unemployment_delta ud USING (departamento)
    JOIN desertion_delta dd USING (departamento)
    WHERE pd.mejora_pobreza > 0
      AND ud.mejora_desempleo > 0
      AND dd.mejora_desercion > 0
    ORDER BY score_compuesto DESC
    LIMIT 10
""").df()

In [ ]:
# Comprehensive department scorecard: CTEs + JOINs + window functions
con.sql("""
    WITH pop_latest AS (
        SELECT departamento, region, poblacion
        FROM poblacion WHERE ano = 2024
    ),
    pov_score AS (
        SELECT departamento,
               ROUND(incidencia_pobreza * 100, 1) AS pobreza_pct,
               ROUND(gini, 3) AS gini,
               PERCENT_RANK() OVER (ORDER BY incidencia_pobreza) AS pctl_pobreza
        FROM pobreza WHERE ano = 2024
    ),
    emp_score AS (
        SELECT departamento,
               ROUND(AVG(tasa_desempleo) * 100, 1) AS desempleo_pct,
               ROUND(AVG(tasa_informalidad) * 100, 1) AS informalidad_pct,
               PERCENT_RANK() OVER (ORDER BY AVG(tasa_desempleo)) AS pctl_desempleo
        FROM empleo WHERE ano = 2023
        GROUP BY departamento
    ),
    edu_score AS (
        SELECT departamento,
               ROUND(AVG(tasa_desercion) * 100, 2) AS desercion_pct,
               PERCENT_RANK() OVER (ORDER BY AVG(tasa_desercion)) AS pctl_desercion
        FROM educacion WHERE ano = 2024
        GROUP BY departamento
    )
    SELECT
        pl.departamento,
        pl.region,
        pl.poblacion,
        ps.pobreza_pct,
        es.desempleo_pct,
        es.informalidad_pct,
        eds.desercion_pct,
        ps.gini,
        -- Composite index: average of percentile ranks (lower = better)
        ROUND((ps.pctl_pobreza + es.pctl_desempleo + eds.pctl_desercion) / 3 * 100, 1) AS indice_bienestar,
        RANK() OVER (ORDER BY (ps.pctl_pobreza + es.pctl_desempleo + eds.pctl_desercion) / 3) AS rank_nacional
    FROM pop_latest pl
    JOIN pov_score ps USING (departamento)
    JOIN emp_score es USING (departamento)
    JOIN edu_score eds USING (departamento)
    ORDER BY rank_nacional
    LIMIT 15
""").df()

## Visualizacion de resultados -- Los numeros cuentan mejor con graficos

SQL nos da los numeros, pero para comunicar hallazgos necesitamos graficos. Lo que hacemos aqui es tomar los resultados de nuestras consultas SQL y pasarlos a matplotlib y seaborn para visualizarlos.

Vamos a crear 4 graficos:
1. Box plot de pobreza por region -- para ver la distribucion y los outliers
2. Tendencia de desempleo por region -- para ver el impacto del COVID y la recuperacion
3. Matriz de correlacion -- para entender como se relacionan los indicadores entre si
4. Ranking de departamentos -- el scorecard final con el indice compuesto

In [ ]:
# Chart 1: Poverty distribution by region (box plot)
df_box = con.sql("""
    SELECT p.region, ROUND(pov.incidencia_pobreza * 100, 1) AS pobreza_pct
    FROM pobreza pov
    JOIN poblacion p USING (departamento, ano)
    WHERE pov.ano = 2024
""").df()

fig, ax = plt.subplots(figsize=(12, 6))
order = df_box.groupby("region")["pobreza_pct"].median().sort_values(ascending=False).index
sns.boxplot(data=df_box, x="region", y="pobreza_pct", order=order, palette="YlOrRd", ax=ax)
ax.set_title("Poverty Incidence by Region (2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Poverty Incidence (%)")
ax.axhline(df_box["pobreza_pct"].median(), color="gray", linestyle="--", alpha=0.7, label="National median")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Unemployment trend by region (line chart)
df_trend = con.sql("""
    SELECT
        p.region,
        e.ano,
        ROUND(AVG(e.tasa_desempleo) * 100, 1) AS avg_desempleo
    FROM empleo e
    JOIN poblacion p USING (departamento, ano)
    GROUP BY p.region, e.ano
    ORDER BY p.region, e.ano
""").df()

fig, ax = plt.subplots(figsize=(12, 6))
for region in df_trend["region"].unique():
    subset = df_trend[df_trend["region"] == region]
    ax.plot(subset["ano"], subset["avg_desempleo"], marker="o", linewidth=2, label=region)

ax.axvspan(2019.8, 2020.5, alpha=0.1, color="red", label="COVID impact")
ax.set_title("Average Unemployment Rate by Region (2019-2023)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Unemployment Rate (%)")
ax.legend(loc="upper right", framealpha=0.9)
ax.set_xticks(range(2019, 2024))
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Correlation heatmap of socioeconomic indicators
df_corr = con.sql("""
    SELECT
        pov.incidencia_pobreza,
        pov.pobreza_extrema,
        pov.gini,
        emp.tasa_desempleo,
        emp.tasa_informalidad,
        edu.tasa_desercion
    FROM pobreza pov
    JOIN (
        SELECT departamento, ano, AVG(tasa_desempleo) AS tasa_desempleo,
               AVG(tasa_informalidad) AS tasa_informalidad
        FROM empleo GROUP BY departamento, ano
    ) emp USING (departamento, ano)
    JOIN (
        SELECT departamento, ano, AVG(tasa_desercion) AS tasa_desercion
        FROM educacion GROUP BY departamento, ano
    ) edu USING (departamento, ano)
""").df()

labels = ["Poverty", "Extreme Poverty", "Gini", "Unemployment", "Informality", "Dropout Rate"]
corr_matrix = df_corr.corr()
corr_matrix.index = labels
corr_matrix.columns = labels

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix: Socioeconomic Indicators", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Department ranking dashboard (top 15 and bottom 15 by composite wellbeing index)
df_rank = con.sql("""
    WITH scores AS (
        SELECT
            pov.departamento,
            PERCENT_RANK() OVER (ORDER BY pov.incidencia_pobreza) AS p_pov,
            PERCENT_RANK() OVER (ORDER BY emp.avg_desemp) AS p_emp,
            PERCENT_RANK() OVER (ORDER BY edu.avg_des) AS p_edu
        FROM pobreza pov
        JOIN (
            SELECT departamento, AVG(tasa_desempleo) AS avg_desemp
            FROM empleo WHERE ano = 2023 GROUP BY departamento
        ) emp USING (departamento)
        JOIN (
            SELECT departamento, AVG(tasa_desercion) AS avg_des
            FROM educacion WHERE ano = 2024 GROUP BY departamento
        ) edu USING (departamento)
        WHERE pov.ano = 2024
    )
    SELECT departamento,
           ROUND((p_pov + p_emp + p_edu) / 3 * 100, 1) AS indice
    FROM scores
    ORDER BY indice
""").df()

fig, ax = plt.subplots(figsize=(12, 9))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(df_rank)))
ax.barh(df_rank["departamento"], df_rank["indice"], color=colors)
ax.set_xlabel("Composite Wellbeing Index (0 = best, 100 = worst)")
ax.set_title("Department Ranking: Composite Socioeconomic Index (2024)", fontsize=14, fontweight="bold")
ax.axvline(50, color="gray", linestyle="--", alpha=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Hallazgos principales

Despues de explorar los datos con SQL, estos son los patrones mas importantes que encontramos:

1. **La desigualdad regional es estructural.** Las regiones de Amazonia y Caribe consistentemente muestran una incidencia de pobreza 15-25 puntos porcentuales por encima de la region Andina, con alta correlacion con las tasas de informalidad laboral.

2. **El impacto del COVID fue asimetrico.** La pobreza subio aproximadamente 7 puntos porcentuales a nivel nacional en 2020, pero los departamentos con mayor pobreza base absorbieron un choque proporcionalmente mayor y se recuperaron mas lento.

3. **Pobreza, informalidad y desigualdad forman un cluster que se refuerza mutuamente.** La matriz de correlacion muestra relaciones positivas fuertes (r > 0.7) entre incidencia de pobreza, informalidad laboral y coeficiente de Gini -- confirmando que las condiciones estructurales del mercado laboral impulsan la desigualdad.

4. **Las tasas de desercion educativa muestran menor variacion regional** comparadas con otros indicadores, lo que sugiere que las politicas educativas a nivel nacional tienen un efecto nivelador entre departamentos.

5. **El indice compuesto revela un gradiente claro.** Bogota, Cundinamarca y Santander consistentemente lideran el ranking, mientras que Choco, La Guajira y Vaupes ocupan las ultimas posiciones -- un patron que persiste en todos los anos analizados.

### Lo que nos llevamos sobre SQL

SQL nos permite explorar datos multidimensionales de manera rapida y estructurada. Con DuckDB, todo el pipeline analitico -- desde JOINs multi-tabla hasta sistemas de ranking con window functions y pivotes condicionales -- corre en memoria sin necesidad de ningun servidor de base de datos.

Lo que practicamos en este notebook cubre el toolkit central de SQL para cualquier rol analitico:
- `JOIN` (inner, left, self, multi-tabla)
- `GROUP BY` con `HAVING`
- Window functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`, `LAG`, `LEAD`, `NTILE`, `PERCENT_RANK`, agregados corridos)
- CTEs (simples, anidados, combinados con window functions)
- Subqueries correlacionadas
- Categorizacion con `CASE WHEN`
- Pivotes con agregacion condicional
- `UNION ALL` para combinar resultados

La razon por la que insisto en SQL es que es la unica herramienta que vas a encontrar en absolutamente todas las empresas que trabajan con datos. Python va y viene segun el equipo, pero SQL siempre esta ahi.